# 🦜🔗 LangChain Fundamentals: A Hands-On Tour
Welcome!  
This notebook gives you a fast-paced, practical tour of the **core components that make LangChain powerful**.  
You’ll learn how to talk to AI, make it use tools, and even make it read your own data.

## 📚 What We Will Cover

### **1. Prompt Templates**
Learn how to communicate with the model efficiently using reusable and structured prompts.
### **2. Chat Models**
These are the *brains* of your application—LLMs that generate responses, reason, and follow instructions.
### **3. LCEL (Chaining)**
Connect multiple steps together to build workflows.  
### **4. Structured Output**
Receive clean, typed outputs (like JSON or Pydantic models) for smooth integration with code.
### **5. Tool Calling**
Giving the AI a calculator, API access, or other utilities. 


Set your OpenAI key (skip if already in your shell env).

In [1]:
from dotenv import load_dotenv
load_dotenv(dotenv_path=".env")

True

In [3]:
# import os
# print(os.getenv("OPENAI_API_KEY"))

## 1) Prompt Templates
Imagine ordering at a subway shop. The sentence structure is always the same: 
*"I would like a **{Size}** sandwich on **{Bread_Type}**."*
You don't invent a new way to speak every time you order lunch; you just change the specific ingredients. Prompt Templates are exactly that—reusable blueprints where we dynamic variables (like "6-inch" or "Italian Herb") without rewriting the whole request.

###  Code Context
In the code below:
* `ChatPromptTemplate`: This is our blueprint.
* `{user_query}` and `{todays_date}`: These are the **variables** (the blanks we will fill in later).
* We use a System Prompt to tell the AI *who* it is (a Date Assistant) before it answers.


![Diagram](./images/01_img.png)

In [2]:
from langchain_prompts import DATE_ASSISTANT_SYSTEM_PROMPT
from datetime import datetime
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", DATE_ASSISTANT_SYSTEM_PROMPT),
    ("human", "{user_query}")
])

example_user_query = """
What is tomorrow's date?
"""

messages = prompt.format_messages(user_query = example_user_query, 
                                    todays_date = datetime.now().strftime("%Y-%m-%d"))  # Shows message objects ready for a chat model

messages


# Other Message Objects
    # System messages
    # Human messages
    # AIMessage
    # ToolMessage
    

[SystemMessage(content="\n\nYou are a helpful and accurate date assistant. \nYou are given a user's request, and today's date. Your job is to accurately infer the date in YYYY-MM-DD format from the user's request.\n\ntoday's date : 2026-08-11\n\nchat_history:\n\n", additional_kwargs={}, response_metadata={}),
 HumanMessage(content="\nWhat is tomorrow's date?\n", additional_kwargs={}, response_metadata={})]

In [3]:
print(messages[0].content)



You are a helpful and accurate date assistant. 
You are given a user's request, and today's date. Your job is to accurately infer the date in YYYY-MM-DD format from the user's request.

today's date : 2026-08-11

chat_history:




## 2) Chat Models (OpenAI)

Think of Chat Models as different brands of TVs (Sony, Samsung, LG). Each one works differently on the inside, but LangChain gives you a **"Universal Remote"** to control them all.
This "Universal Remote" is called the **Runnable Interface**. It means you can press the "Play" button (`invoke`) on *any* model, and it will work exactly the same way. You don't need to learn a new remote control just because you bought a new TV.

### 🧠 Interface Concepts
all Chat Models implement the **Runnable Interface**. This guarantees they all speak the same language:
1.  **Standard Inputs:** They all accept a list of `Messages` (System, Human, AI).
2.  **Standard Outputs:** They all return an `AIMessage`.
3.  **Standard Methods:**
    * `.invoke()`: Send a message and wait for the full answer.
    * `.stream()`: Get the answer word-by-word (like a typewriter).
    * `.batch()`: Send 50 questions at once and get 50 answers back.

###  Code Context
In the code below:
* `init_chat_model`: A universal function to load a model.
* `model="gpt-4o-mini"`: We are selecting a specific, fast model from OpenAI.
* `temperature=0`: We set this to 0 to make the AI **factual and consistent**. Higher numbers (up to 1) make it more creative and random.

- Supports over 100 model providers : https://docs.langchain.com/oss/python/integrations/chat
- https://docs.langchain.com/oss/python/langchain/models

![Diagram](./images/02_img.png)

In [4]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(model="gpt-4o-mini", temperature=0) # Tip: Try other models/model providers

## 3) LLM Invocation  

* **Invoke:** This is like sending a text message and waiting for the reply. You don't see anything until the whole message arrives.
* **Stream:** This is like watching someone type in real-time. You see the words appear one by one. This feels much faster to a user.

###  Code Context
In the code below:
* `llm.invoke(messages)`: Sends our formatted prompt to the AI and waits for the full `AIMessage` response.
* `llm.stream(messages)`: Returns a generator that prints tokens as they arrive.

📚 **[Docs](https://docs.langchain.com/oss/python/langchain/models#invocation)**

![Diagram](./images/03_img.png)

In [5]:
# Invoke : 
response = llm.invoke(messages)
response


AIMessage(content="Tomorrow's date is 2026-08-12.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 73, 'total_tokens': 85, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a2b728f9dd', 'id': 'chatcmpl-EBdMWblF3ZuWofqs2ncHBRcZKGE6W', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ff037-e18d-7db1-aab9-549573c843d8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 73, 'output_tokens': 12, 'total_tokens': 85, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [6]:
print(response.content)

Tomorrow's date is 2026-08-12.


## 3.1) Streaming (Real-time Output)

Imagine you hired a ghostwriter to write a story.
* **Invoke (Standard):** The writer goes into a private room, locks the door, writes the entire story, and comes out 20 minutes later to hand you the finished manuscript. You sit in silence waiting the whole time.
* **Stream (Real-time):** The writer sits right in front of you and types. You see every letter appear on the screen the moment they press the key. The story takes the same amount of time to finish, but it **feels instant** because you are engaged immediately.

### Technical Concept
LLMs don't "think" of a whole sentence at once; they generate text **one token (word part) at a time**.
* **The Bottleneck:** By default (`.invoke()`), the program buffers (holds) all these tokens in memory until the model is completely finished. This creates **Latency** (the delay between asking and seeing the answer).
* **The Solution:** The `.stream()` method bypasses this buffer. It creates a Python **Generator** that yields each token (`AIMessageChunk`) the exact millisecond it is created. This drastically lowers the "Time to First Token."

### Code Context
In the code below:
* `llm.stream(messages)`: This replaces `.invoke()`. It returns an iterable stream, not a final string.
* `print(..., end="", flush=True)`: We force Python to print *immediately* without moving to a new line, creating that smooth "typewriter" effect.

📚 **[Docs: Streaming](https://docs.langchain.com/oss/python/langchain/streaming)**


In [7]:
# Streaming : 

for chunk in llm.stream(messages):
    print(chunk.content, end="", flush=True)


Tomorrow's date is 2026-08-12.

## 4) Chaining with LCEL (LangChain Expression Language)

Imagine a factory assembly line:
* **Station A:** Puts dough on the belt (The Prompt).
* **Station B:** Bakes the dough (The Model).
* **Station C:** Packages the bread (The Output Parser).

LCEL lets us pipe (`|`) these steps together. The output of one step automatically becomes the input of the next.

###  Code Context
In the code below:
* `chain = prompt | llm | StrOutputParser()`
* The `|` symbol is the magic pipe.
* `StrOutputParser`: Converts the complex AI message object directly into a simple string so we don't have to do `print(response.content)`.

📚 **[Docs: LCEL Conceptual Guide](https://python.langchain.com/docs/concepts/lcel/)** *(the LCEL guide still lives on the legacy docs site — in LangChain 1.x, piping is fully supported for simple prompt → model → parser chains, but agents are the recommended pattern for anything that loops or branches)*

![Diagram](./images/04_img.png)

In [8]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | llm | StrOutputParser()


print(chain.invoke({ 'user_query' : example_user_query, 
                    'todays_date' : datetime.now().strftime("%Y-%m-%d") }))


Tomorrow's date is 2026-08-12.


## 5) Structured Output

Usually, AI likes to chat and write paragraphs. But sometimes, you want a form filled out, not a conversation.
If you ask for a date range, you don't want *"Sure! The start date is..."*. You want `{"start": "2025-01-01", "end": "2025-01-05"}`.
Structured output forces the AI to stop chatting and generate data matching a specific schema.

###  Code Context
In the code below:
* `class DateRange(BaseModel)`: We define the "Form" we want the AI to fill out using Pydantic.
* `llm.with_structured_output(DateRange)`: We tell the LLM, "Your output **must** match this class."
* The result is an actual Python object, not a string.

📚 **[Docs: Structured Output](https://docs.langchain.com/oss/python/langchain/models#structured-output)**

![Diagram](./images/05_img.png)

In [9]:
# Supports
    # - pydantic
    # - Typedict
    # - Dataclass 
    # - Json schema

In [10]:
from typing import List, Optional, TypedDict
from pydantic import BaseModel, Field
from langchain_prompts import DATE_EXTRACTOR_PROMPT_TEMPLATE

# # You can define structure with Pydantic
# class DateRange(BaseModel):
#     start_date: str = Field(default = None, description="Start Date of the date range")
#     end_date: str = Field(default = None, description="End Date of the date range")

# OR you can define structure with TypedDict
class DateRange(TypedDict):
    start_date: str
    end_date: str


llm = init_chat_model(model="gpt-4o-mini", temperature=0)

structured_llm = llm.with_structured_output(DateRange)

prompt = ChatPromptTemplate.from_messages([
    ("system", DATE_EXTRACTOR_PROMPT_TEMPLATE),
    ("human", "{user_query}")
])


user_query = "I need time off for the next 4 days"
todays_date = datetime.now().strftime("%Y-%m-%d")


chain = prompt | structured_llm

llm_response = chain.invoke({'todays_date': todays_date, 'user_query': user_query})

llm_response


{'start_date': '2026-08-12', 'end_date': '2026-08-15'}

## 6) Tool Calling

LLMs are great at writing poetry, but they are often bad at math and they don't know the current weather.
Tool calling is like giving the AI **specialized gadgets** to help it do its job.
You tell the AI: *"I am giving you a calculator tool. If you see a math problem, don't guess—use the tool."*
The AI decides **when** it needs help and **which** gadget to pick to solve the problem.

###  Code Context
In the code below:
* `@tool`: This is a decorator. It puts a "sticker" on your Python function that says *"Hey AI, you are allowed to use this!"*
* `llm.bind_tools(tools)`: This effectively hands the manual of available tools to the "Brain."
* **Important:** When the model uses a tool, it doesn't return a text answer immediately. It returns a `tool_call` (a request to run the code). You (the code) must then run that function and give the answer back to the AI.

📚 **[Docs: Tool Calling](https://docs.langchain.com/oss/python/langchain/models#tool-calling)**


![Diagram](./images/06_img.png)

In [11]:
from langchain.tools import tool
from langchain.messages import HumanMessage
import math

@tool
def circle_area(radius: float) -> float: 
    """Return the area of a circle for a given radius."""
    return math.pi * radius * radius

tools = [circle_area]

# Bind tools to the model so it can decide to call them
llm_with_tools = llm.bind_tools(tools)

msg = HumanMessage(content="I have a circular garden of radius 3. What's the area?")
ai_msg = llm_with_tools.invoke([msg])

print("Model output Type", type(ai_msg))
print("Model output", ai_msg)
print("Tool calls (if any):", getattr(ai_msg, "tool_calls", None))

Model output Type <class 'langchain_core.messages.ai.AIMessage'>
Model output content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 62, 'total_tokens': 76, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_5691373d70', 'id': 'chatcmpl-EBdMxnK5diWzrQrNcSuu63hBY2c7y', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019ff038-5333-7253-a5db-bef287064a4e-0' tool_calls=[{'name': 'circle_area', 'args': {'radius': 3}, 'id': 'call_OdlCyT8fYuTQJH4YCa6B0Mwt', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 62, 'output_tokens': 14, 'total_tokens': 76, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_toke

In [12]:
# If a tool call is present, execute it and return a final answer:

final = None
if getattr(ai_msg, "tool_calls", None):
    for tool_call in ai_msg.tool_calls:
        if tool_call["name"] == "circle_area":
            r = float(tool_call["args"]["radius"])
            result = circle_area.invoke({"radius": r})
            final = f"The area is approximately {result:.2f} square units."
else:
    final = ai_msg.content

print("Final:", final)

Final: The area is approximately 28.27 square units.


## 7) Create Agent

In Section 6, *you* were the loop — the AI flagged a tool call, you ran it, and handed the result back. That works for one call. But what if the question needs two tools? You'd have to build that dispatch logic yourself.

`create_agent` handles that loop for you. Every agent is two things:

**Agent = Model + Harness**
* **Model** — the brain. Reads context and decides: *call a tool, or give the final answer.*
* **Harness** — the loop. Its job is to *get the model the right context at the right time* — run the requested tool, feed the result back, and repeat until the model is done.

![Agent loop](./agent_loop.png)

### Code Context
In the code below:
* `create_agent`: Builds the agent — takes a `model`, a list of `tools`, and an optional `system_prompt`.
* `tools=[circle_area, rectangle_area]`: We give the agent **two** tools. It decides which ones to call and when.
* `agent.invoke({"messages": [...]})`: The input is a dict with a `"messages"` key — not a raw list.
* `result["messages"][-1].content`: The agent returns all messages (question → tool calls → tool results → final answer). The last one is the answer.

📚 **[Docs: Agents](https://docs.langchain.com/oss/python/langchain/agents)**

In [13]:
from langchain.agents import create_agent
from langchain.tools import tool
import math

@tool
def circle_area(radius: float) -> float:
    """Return the area of a circle for a given radius.

    Args:
        radius: The radius of the circle
    """
    return math.pi * radius * radius

@tool
def rectangle_area(length: float, width: float) -> float:
    """Return the area of a rectangle given its length and width.

    Args:
        length: The length of the rectangle
        width: The width of the rectangle
    """
    return length * width

agent = create_agent(
    model=llm,
    tools=[circle_area, rectangle_area],
    system_prompt="You are a helpful geometry assistant. Always show your calculations."
)

In [14]:
question = "I have a circular pond with radius 3 and a rectangular patio that is 4 by 5. What is the combined area of both?"

result = agent.invoke({"messages": [{"role": "user", "content": question}]})

print("Final Answer:", result["messages"][-1].content)

# Peek under the hood — the agent called both tools automatically
print("\n--- Agent's steps ---")
for msg in result["messages"]:
    label = type(msg).__name__
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print(f"{label}: called → {[tc['name'] for tc in msg.tool_calls]}")
    elif msg.content:
        print(f"{label}: {str(msg.content)[:80]}")

Final Answer: To find the combined area of the circular pond and the rectangular patio, we first calculate the area of each:

1. **Area of the circular pond**:
   \[
   \text{Area} = \pi \times r^2 = \pi \times 3^2 \approx 28.27 \text{ square units}
   \]

2. **Area of the rectangular patio**:
   \[
   \text{Area} = \text{length} \times \text{width} = 4 \times 5 = 20 \text{ square units}
   \]

Now, we add both areas together:
\[
\text{Combined Area} = 28.27 + 20 = 48.27 \text{ square units}
\]

Thus, the combined area of the circular pond and the rectangular patio is approximately **48.27 square units**.

--- Agent's steps ---
HumanMessage: I have a circular pond with radius 3 and a rectangular patio that is 4 by 5. Wha
AIMessage: called → ['circle_area', 'rectangle_area']
ToolMessage: 28.274333882308138
ToolMessage: 20.0
AIMessage: To find the combined area of the circular pond and the rectangular patio, we fir


![Diagram](./images/07_img.png)

In [15]:
## Simple Q/A Chatbot
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=llm,
    tools=[circle_area, rectangle_area],
    system_prompt="You are a helpful geometry assistant. Always show your calculations.",
    checkpointer=InMemorySaver()
)

config = {"configurable": {"thread_id": "geometry-chat"}}

print("Geometry Chatbot ready! Type 'quit' to exit.\n")

while True:
    user_input = input("You: ").strip()
    if not user_input:
        continue
    if user_input.lower() in ("quit", "exit"):
        print("Goodbye!")
        break

    result = agent.invoke(
        {"messages": [{"role": "user", "content": user_input}]},
        config=config
    )
    print(f"Agent: {result['messages'][-1].content}\n")

Geometry Chatbot ready! Type 'quit' to exit.

Agent: The area of a circle can be calculated using the formula:

\[
\text{Area} = \pi r^2
\]

where \( r \) is the radius of the circle and \( \pi \) (pi) is approximately 3.14159. 

If you provide the radius, I can calculate the area for you.

Agent: The area of a circle with a radius of 1.5 is approximately \( 7.07 \) square units.

Goodbye!
